In [1]:
# 1. Install Linux OS tools (aria2)
!sudo apt-get update && sudo apt-get install aria2 -y

# 2. Install all Python packages and upgrade/pin specific versions in one go
!pip install -U crawl4ai playwright-stealth browserforge "crawlee[beautifulsoup,curl-impersonate,playwright]" urllib3==1.26.18 requests

# 3. Setup the browser engines
!crawl4ai-setup
!playwright install chromium
!playwright install-deps chromium

Get:1 https://cli.github.com/packages stable InRelease [3917 B]
Get:2 https://download.docker.com/linux/ubuntu noble InRelease [48.5 kB]       
Get:3 https://packages.cloud.google.com/apt cloud-sdk InRelease [1621 B]       
Get:4 https://cli.github.com/packages stable/main amd64 Packages [354 B]       
Get:5 https://download.docker.com/linux/ubuntu noble/stable amd64 Packages [60.5 kB]
Hit:6 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble InRelease          
Get:7 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:8 https://packages.cloud.google.com/apt cloud-sdk/main all Packages [1988 kB]
Get:9 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:10 https://cloud.archive.ubuntu.com/ubuntu noble InRelease [256 kB]        
Get:11 https://packages.cloud.google.com/apt cloud-sdk/main amd64 Packages [4659 kB]
Get:12 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [2402 kB]
Get:13 ht

In [ ]:
#!/bin/bash

echo "[*] Killing old Tor instances..."
sudo killall tor 2>/dev/null || true

BASE_DIR="$HOME/.tor_pool"
mkdir -p "$BASE_DIR"

echo "[*] Spinning up 100 Tor proxy instances with HTTP Tunneling..."

for i in {1..100}
do
    # Calculate ports
    SOCKS_PORT=$((9050 + i))
    HTTP_PORT=$((8050 + i)) # <--- HTTP PORTS 8051 TO 8150
    
    INSTANCE_DIR="$BASE_DIR/data_$SOCKS_PORT"
    mkdir -p "$INSTANCE_DIR"
    
    # Create the config file (torrc)
    CONFIG_FILE="$BASE_DIR/torrc_$SOCKS_PORT"
    cat <<EOF > "$CONFIG_FILE"
SocksPort 127.0.0.1:$SOCKS_PORT
HTTPTunnelPort 127.0.0.1:$HTTP_PORT
DataDirectory $INSTANCE_DIR
RunAsDaemon 1
Log notice file $INSTANCE_DIR/notice.log
EOF

    # Start the Tor instance
    tor -f "$CONFIG_FILE"
    echo "[+] Started Tor on SOCKS:$SOCKS_PORT | HTTP:$HTTP_PORT"
done

echo "[*] All 100 Tor proxies are running!"

In [2]:
# %%
# ==========================================
# CELL 1: THE INDEX PARSER (STEALTH + PATIENCE)
# ==========================================
import json
import asyncio
import nest_asyncio
from playwright.async_api import async_playwright
from playwright_stealth import Stealth

nest_asyncio.apply()

async def build_index_checkpoint():
    url = "https://rarelust.com/movies-index/"
    
    # Let's use port 8055 to get a completely fresh Tor IP
    target_proxy = "http://127.0.0.1:8055"
    print(f"[*] Fetching Master Index via Tor Proxy ({target_proxy})...")
    
    async with Stealth().use_async(async_playwright()) as p:
        browser = await p.chromium.launch(
            headless=True,
            proxy={"server": target_proxy},
            args=[
                "--no-sandbox",             
                "--disable-dev-shm-usage",  
                "--disable-gpu"             
            ]
        )
        
        context = await browser.new_context(
            user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
            viewport={'width': 1920, 'height': 1080},
            locale='en-US',
            timezone_id='America/New_York'
        )
        page = await context.new_page()
        
        print("[*] Waiting for WAF/Cloudflare challenge to clear...")
        try:
            # BUMPED TIMEOUT TO 120 SECONDS (Tor is slow!)
            await page.goto(url, timeout=120000)
            await page.wait_for_selector('div.entry-content', timeout=120000)
            print("[+] Challenge cleared! Waiting for the massive list to fully load...")
            
            # 1. Wait for the network to stop downloading background scripts/images
            try:
                await page.wait_for_load_state('networkidle', timeout=15000)
            except Exception:
                pass 
                
            # 2. Scroll to the bottom to trigger any lazy-loaded HTML
            print("[*] Scrolling to bottom...")
            await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
            
            # 3. Hard pause for 10 seconds to let the browser parse all 20,000 DOM elements
            print("[*] Sleeping for 10 seconds to let the DOM render...")
            await page.wait_for_timeout(10000)
            
            print("[+] Extracting links...")
        except Exception as e:
            print(f"[-] Failed to bypass challenge: {e}")
            await page.screenshot(path="debug_cloudflare_block.png")
            print("[!] Saved screenshot of the block to 'debug_cloudflare_block.png'.")
            await browser.close()
            return

        # Execute JavaScript in the browser to extract all hrefs
        links = await page.eval_on_selector_all(
            'div.entry-content li a',
            'elements => elements.map(el => el.href)'
        )
        
        await browser.close()

    # Clean up empty links, filter out category pages, deduplicate, and sort
    unique_links = sorted(list(set([
        link for link in links 
        if link and "/category/" not in link.lower()
    ])))
    
    checkpoint_file = 'rarelust_target_urls.json'
    with open(checkpoint_file, 'w', encoding='utf-8') as f:
        json.dump(unique_links, f, indent=4)
        
    print(f"[+] SUCCESS! {len(unique_links)} unique movie URLs saved to '{checkpoint_file}'")

# Run the cell
await build_index_checkpoint()

[*] Fetching Master Index via Tor Proxy (http://127.0.0.1:8055)...
[*] Waiting for WAF/Cloudflare challenge to clear...
[+] Challenge cleared! Waiting for the massive list to fully load...
[*] Scrolling to bottom...
[*] Sleeping for 10 seconds to let the DOM render...
[+] Extracting links...
[+] SUCCESS! 20734 unique movie URLs saved to 'rarelust_target_urls.json'


In [ ]:
# %%
# ==========================================
# CELL 2: PLAYWRIGHT TOR SCRAPER (100 CONCURRENCY)
# ==========================================
import asyncio
import json
import logging
import os
import nest_asyncio
from datetime import datetime, timedelta

from crawlee import ConcurrencySettings
from crawlee.crawlers import PlaywrightCrawler, PlaywrightCrawlingContext
from crawlee.proxy_configuration import ProxyConfiguration

nest_asyncio.apply()

# Muzzle all of Crawlee's internal console logs and storage warnings
logging.getLogger('crawlee').setLevel(logging.ERROR)
logging.getLogger('crawlee._autoscaling.autoscaled_pool').setLevel(logging.ERROR)
logging.getLogger('crawlee.storage_clients').setLevel(logging.ERROR)

def log_to_file(msg):
    timestamp = datetime.now().strftime("%H:%M:%S")
    with open("spider.log", "a", encoding="utf-8") as f:
        f.write(f"[{timestamp}] {msg}\n")

async def scrape_movies():
    checkpoint_file = 'rarelust_target_urls.json'
    database_file = 'movies_database.jsonl'
    
    # 1. LOAD TARGET URLs
    try:
        with open(checkpoint_file, 'r', encoding='utf-8') as f:
            target_urls = json.load(f)
    except FileNotFoundError:
        log_to_file(f"[-] Error: Could not find {checkpoint_file}. Run Cell 1 first.")
        return

    # 2. CHECK EXISTING STATE (Resume Logic)
    scraped_urls = set()
    if os.path.exists(database_file):
        with open(database_file, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    data = json.loads(line.strip())
                    scraped_urls.add(data['url'])
                except json.JSONDecodeError:
                    continue 
                    
    # Filter out URLs we have already successfully scraped
    pending_urls = [url for url in target_urls if url not in scraped_urls]
    
    if not pending_urls:
        log_to_file("[+] All URLs have already been scraped! Nothing to do.")
        return
        
    log_to_file(f"[*] Resuming extraction for the remaining {len(pending_urls)} URLs...")

    # ==========================================
    # 3. PROXY & CRAWLER SETUP
    # ==========================================
    # Pointing to Tor's native HTTPTunnelPorts (8051 through 8150)
    tor_proxies = [f"http://127.0.0.1:{port}" for port in range(8051, 8151)]
    
    proxy_config = ProxyConfiguration(proxy_urls=tor_proxies)

    # Wrap the concurrency limit in the proper Python object
    # SET TO 100 FOR THE 96-CORE MACHINE
    concurrency_settings = ConcurrencySettings(max_concurrency=100)

    # Initialize Playwright 
    crawler = PlaywrightCrawler(
        max_requests_per_crawl=len(pending_urls),
        max_request_retries=3,
        proxy_configuration=proxy_config,
        concurrency_settings=concurrency_settings, 
        request_handler_timeout=timedelta(seconds=60), 
        
        # --- MAXIMUM CPU SAVINGS MODE ---
        browser_launch_options={
            "args": [
                "--no-sandbox",             
                "--disable-dev-shm-usage",  
                "--disable-gpu",
                "--disable-background-networking", 
                "--disable-extensions",            
                "--mute-audio"                     
            ]
        }
    )

    @crawler.router.default_handler
    async def movie_handler(context: PlaywrightCrawlingContext):
        page = context.page
        url_name = context.request.url.rstrip('/').split('/')[-1]
        
        # SAFETY NET: Ignore category index pages entirely
        if "/category/" in context.request.url:
            log_to_file(f"[-] Skipping category page: {url_name}")
            return
            
        log_to_file(f"[>] Hitting: {url_name} (Waiting for WAF/Cloudflare...)")
        
        try:
            await page.wait_for_selector('div.entry-content', timeout=45000)
        except Exception:
            log_to_file(f"[!] Timeout/Challenge Failed on: {url_name}. Retrying later.")
            raise Exception("Challenge Timeout") 
            
        log_to_file(f"[+] Challenge cleared! Extracting: {url_name}")
        
        # EXTRACT DATA USING PLAYWRIGHT LOCATORS
        title_element = page.locator('h1.entry-title')
        title = await title_element.inner_text() if await title_element.count() > 0 else "Unknown Title"
        
        # --- THE FIX: Target ALL entry-content blocks and stitch them together ---
        content_locators = page.locator('div.entry-content')
        all_texts = await content_locators.all_inner_texts()
        page_text = "\n\n".join(all_texts)
        
        # Get all links from ALL content blocks
        all_links_elements = await content_locators.locator('a').all()
        all_links = []
        for a in all_links_elements:
            href = await a.get_attribute('href')
            if href:
                all_links.append(href)
        
        # EXCLUSION LIST
        junk_domains = ['rarelust.com', 'imdb.com', 'youtube.com', 'youtu.be', 't.me', 'twitter.com', 'facebook.com']
        download_links = list(set([
            link for link in all_links 
            if not any(junk in link.lower() for junk in junk_domains)
        ]))

        # SAVE EXTRACTED DATA
        if download_links:
            movie_data = {
                'url': context.request.url,
                'title': title,
                'download_links': download_links,
                'metadata': page_text 
            }
            with open(database_file, 'a', encoding='utf-8') as f:
                f.write(json.dumps(movie_data) + '\n')
            
            with open('aria2_downloads.txt', 'a', encoding='utf-8') as f:
                for link in download_links:
                    f.write(f"{link}\n")
    
    # 4. RUN CRAWLER WITH FILTERED LIST
    try:
        await crawler.run(pending_urls) 
    except (KeyboardInterrupt, asyncio.CancelledError):
        log_to_file("[!] Crawl manually interrupted (Ctrl+C). Shutting down safely...")
    except Exception as e:
        log_to_file(f"[-] Crawl stopped due to an error: {e}")
    finally:
        log_to_file("[+] Scraping process ended! Check movies_database.jsonl and aria2_downloads.txt")

# Run the cell
await scrape_movies()

In [ ]:
chmod +x start_tor_pool.sh 

In [ ]:
sudo apt-get update && sudo apt-get install tor -y

In [ ]:
./start_tor_pool.sh

In [ ]:
rm -rf /teamspace/studios/this_studio/storage

In [ ]:
tail -f spider.log